# OIBSIP Task 1: Iris Flower Classification

End-to-end multiclass classification workflow. This notebook includes data quality checks, exploratory analysis, five tuned models, held-out evaluation and artifact export.

Dataset source: canonical Iris dataset distributed with scikit-learn.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import joblib
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET = "species"
ROOT = Path.cwd()

## 2. Load data and validate quality

In [ ]:
df = pd.read_csv(ROOT / "data" / "iris.csv")
assert df.shape == (150, 5)
assert df[FEATURES].notna().all().all()
assert df[TARGET].notna().all()
quality = pd.DataFrame({
    "check": ["rows", "columns", "missing cells", "duplicate rows", "class count"],
    "result": [len(df), df.shape[1], int(df.isna().sum().sum()),
               int(df.duplicated().sum()), df[TARGET].nunique()]
})
display(quality)
display(df.head())

## 3. Exploratory data analysis

In [ ]:
display(df.describe().T)
display(df.groupby(TARGET)[FEATURES].mean().round(2))
display(df[TARGET].value_counts().rename("observations"))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, feature in zip(axes.flat, FEATURES):
    sns.histplot(data=df, x=feature, hue=TARGET, kde=True, element="step", ax=ax)
    ax.set_title(feature.replace("_", " ").title())
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df, vars=FEATURES, hue=TARGET, corner=True)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[FEATURES].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()

## 4. Stratified train-test split

In [ ]:
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
display(y_train.value_counts(normalize=True).sort_index())

## 5. Five-model comparison and hyperparameter tuning

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidates = {
    "Logistic Regression": (Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=3000))]),
                            {"model__C": [0.1, 1, 10]}),
    "K-Nearest Neighbors": (Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
                            {"model__n_neighbors": [3, 5, 7, 9], "model__weights": ["uniform", "distance"]}),
    "Support Vector Machine": (Pipeline([("scaler", StandardScaler()), ("model", SVC())]),
                               {"model__C": [0.1, 1, 10], "model__kernel": ["linear", "rbf"]}),
    "Random Forest": (RandomForestClassifier(random_state=42),
                      {"n_estimators": [100, 250], "max_depth": [None, 3, 5], "min_samples_leaf": [1, 2]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=42),
                          {"n_estimators": [50, 100], "learning_rate": [0.03, 0.1], "max_depth": [1, 2]})
}
searches, records = {}, []
for name, (estimator, params) in candidates.items():
    search = GridSearchCV(estimator, params, scoring="accuracy", cv=cv, n_jobs=-1, refit=True)
    search.fit(X_train, y_train)
    searches[name] = search
    pred = search.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, pred, average="macro", zero_division=0
    )
    records.append({
        "model": name, "cv_accuracy": search.best_score_,
        "test_accuracy": accuracy_score(y_test, pred),
        "macro_precision": precision, "macro_recall": recall, "macro_f1": f1,
        "best_parameters": str(search.best_params_)
    })
results = pd.DataFrame(records).sort_values("cv_accuracy", ascending=False).reset_index(drop=True)
display(results)

## 6. Final held-out evaluation

In [ ]:
best_name = results.loc[0, "model"]
best_model = searches[best_name].best_estimator_
y_pred = best_model.predict(X_test)
print("Selected model:", best_name)
print(classification_report(y_test, y_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", xticks_rotation=30)
plt.title(f"Confusion Matrix: {best_name}")
plt.tight_layout()
plt.show()

## 7. Feature importance where supported

In [ ]:
model_core = best_model.named_steps["model"] if isinstance(best_model, Pipeline) else best_model
if hasattr(model_core, "feature_importances_"):
    importance = pd.Series(model_core.feature_importances_, index=FEATURES).sort_values()
    display(importance.to_frame("importance"))
    importance.plot(kind="barh", title="Feature importance")
    plt.tight_layout()
    plt.show()
elif hasattr(model_core, "coef_"):
    st = pd.DataFrame(model_core.coef_, columns=FEATURES, index=model_core.classes_)
    display(st)

## 8. Save model and results

In [ ]:
(ROOT / "models").mkdir(exist_ok=True)
(ROOT / "outputs").mkdir(exist_ok=True)
joblib.dump(best_model, ROOT / "models" / "notebook_best_model.joblib")
results.to_csv(ROOT / "outputs" / "notebook_model_comparison.csv", index=False)
(ROOT / "outputs" / "notebook_classification_report.txt").write_text(
    classification_report(y_test, y_pred, zero_division=0), encoding="utf-8"
)
print("Artifacts saved successfully.")

## 9. Findings and limitations

- The dataset is balanced, with 50 samples per species.
- Cross-validation is confined to the training partition.
- The final test partition is not used for hyperparameter selection.
- The test set has 30 rows, so one error changes accuracy by about 3.33 percentage points.
- This curated benchmark does not prove performance on other regions, devices or measurement protocols.
- Model probability scores are not guaranteed to be calibrated.